# Artículo: Limpieza de datos de homicidios
* **Nombre**: Sierra Wells  
* **Clase**: Programación (Geointeligencia Computacional)
* **Programa**: Maestría en Ciencias de Información Geoespacial
* **Fecha**: 19 noviembre 2026

## 📝 Descripción
Lee los microdatos de defunciones registradas del INEGI (estadísticas de defunciones registradas, EDR) registradas de 2018 a 2024, selecciona, codifica y cuenta únicamente los homicidios **ocurridos en 2018** para cada **municipio** del país.

Los municipios son los del censo 2010: los homicidios ocurridos en municipios creados después de 2010 se asignan a su municipio padre, según `clean/hand/mun_crosswalk.yaml`.

Fuente: [INEGI, microdatos de mortalidad](https://www.inegi.org.mx/programas/edr/#microdatos).

Resultado en `clean/output/`:
* `1_clean_homicides.csv`: homicidios por municipio (`cve_mun`), solo de los municipios con al menos un homicidio.

In [1]:
# Librerías
from pathlib import Path

import pandas as pd
import pyogrio
import yaml

In [2]:
# Rutas (el notebook se corre desde clean/src/)
TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / 'input'
HAND_DIR = TASK_DIR / 'hand'
OUTPUT_DIR = TASK_DIR / 'output'

In [3]:
# Año de ocurrencia que se conserva y años de registro de las bases que se leen
YEAR = 2018
REGISTRATION_YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Variables que se leen de las bases de defunciones (además del tipo de defunción)
COLUMNS = ['ENT_OCURR', 'MUN_OCURR', 'ANIO_OCUR']

**1. Leer datos**

El tipo de defunción (accidente, homicidio, suicidio, etc.) se llama `PRESUNTO` en las bases de 2018 a 2021, y `TIPO_DEFUN` a partir de 2022. Por eso se lee una u otra según el año y se renombra a `TIPO_DEFUN`.

In [4]:
# Las bases son por año de registro: algunos homicidios ocurridos en 2018 se registraron en años posteriores,
# así que se leen las bases de 2018 a 2024 y se juntan en una sola tabla
deaths_by_year = []
for registration_year in REGISTRATION_YEARS:
    # El nombre del archivo cambia entre años (DEFUN18.DBF, DEFUN20.dbf), así que se busca sin importar mayúsculas
    dbf_path = [path for path in INPUT_DIR.iterdir() if path.name.upper() == f'DEFUN{str(registration_year)[2:]}.DBF'][0]

    type_column = 'PRESUNTO' if registration_year <= 2021 else 'TIPO_DEFUN'
    deaths_year = pyogrio.read_dataframe(dbf_path, columns=COLUMNS + [type_column], read_geometry=False)
    deaths_year = deaths_year.rename(columns={'PRESUNTO': 'TIPO_DEFUN'})

    # pyogrio no avisa si falta una columna, así que lo revisamos
    assert list(deaths_year.columns) == COLUMNS + ['TIPO_DEFUN'], f'Faltan columnas en {registration_year}'

    # Agregar el año de registro (el año de la base)
    deaths_year['ANIO_REGIS'] = registration_year
    deaths_by_year.append(deaths_year)

deaths = pd.concat(deaths_by_year, ignore_index=True)

# Nombres de columnas en minúsculas
deaths.columns = deaths.columns.str.lower()

In [5]:
deaths.head()

,ent_ocurr,mun_ocurr,anio_ocur,tipo_defun,anio_regis
0,01,001,2018,8,2018
1,01,001,2018,8,2018
2,01,001,2018,8,2018
3,01,001,2018,8,2018
4,01,001,2018,8,2018


**2. Seleccionar homicidios**

En todos los años, el valor 2 de `tipo_defun` es "Homicidio". Según la documentación de las bases, los demás valores cambian a partir de 2022:
* 2018 a 2021 (`PRESUNTO`): 1 accidente, **2 homicidio**, 3 suicidio, 4 se ignora, 5 operaciones legales y de guerra, 8 no aplica para muerte natural.
* 2022 a 2024 (`TIPO_DEFUN`): 1 accidente, **2 homicidio (agresión)**, 3 suicidio, 4 enfermedad (muerte natural), 5 intervención legal, 9 se ignora.

Como solo usamos el valor 2, el cambio no afecta la selección de homicidios.

In [6]:
# Defunciones por tipo de defunción y año de registro
pd.crosstab(deaths['tipo_defun'], deaths['anio_regis'])

anio_regis,2018,2019,2020,2021,2022,2023,2024
tipo_defun,,,,,,,
1,34591,33525,32364,34613,37457,40282,39929
2,36687,36662,36773,35700,33291,32253,33550
3,6808,7225,7896,8433,8241,9072,9051
4,5567,6399,5580,5898,762773,715751,734390
5,96,71,47,54,35,29,82
8,638862,663902,1004083,1037551,0,0,0
9,0,0,0,0,5919,2482,2670


In [7]:
# Seleccionar solo homicidios
homicides = deaths.loc[deaths['tipo_defun'] == 2].drop(columns='tipo_defun')
homicides.head()

,ent_ocurr,mun_ocurr,anio_ocur,anio_regis
200,01,001,2018,2018
438,01,001,2018,2018
957,01,001,2018,2018
970,01,001,2018,2018
1117,01,003,2018,2018


**3. Filtrar por año de ocurrencia + estado y municipio especificados**

Según la documentación de las bases de defunciones del INEGI:
* `anio_ocur`: año en que ocurrió la defunción; 9999 es "No especificado" en todos los años.
* `ent_ocurr`: 01 a 32 son las entidades federativas.
* `mun_ocurr`: clave de 3 dígitos del municipio dentro de la entidad; 999 es "No especificado".

Seleccionamos únicamente homicidios que cumplan con **todas** las siguientes condiciones:
* Ocurrieron en 2018 (`anio_ocur`), aunque se hayan _registrado_ después
* Tienen el estado de ocurrencia (`ent_ocurr`) definido
* Tienen el municipio de ocurrencia (`mun_ocurr`) definido


In [8]:
# Codificamos NA en las tres variables de interés
homicides['anio_ocur'] = homicides['anio_ocur'].astype('Int64').replace(9999, pd.NA)
homicides['ent_ocurr'] = homicides['ent_ocurr'].where(homicides['ent_ocurr'].astype(int) <= 32)
homicides['mun_ocurr'] = homicides['mun_ocurr'].replace('999', pd.NA)

In [9]:
# Filtramos para las tres condiciones mencionadas
in_year = (homicides['anio_ocur'] == YEAR).fillna(False)
has_ent = homicides['ent_ocurr'].notna()
has_mun = homicides['mun_ocurr'].notna()
homicides_clean = homicides.loc[in_year & has_ent & has_mun].copy()

# Clave de municipio (5 dígitos: 2 de la entidad y 3 del municipio)
homicides_clean['cve_mun'] = homicides_clean['ent_ocurr'] + homicides_clean['mun_ocurr']
homicides_clean = homicides_clean[['cve_mun', 'anio_regis']].reset_index(drop=True)

# Revisión: todas las claves de municipio tienen 5 caracteres
assert homicides_clean['cve_mun'].str.len().eq(5).all()

print(f'Homicidios ocurridos en {YEAR} con entidad y municipio especificados: {len(homicides_clean):,}')
homicides_clean.head()

Homicidios ocurridos en 2018 con entidad y municipio especificados: 34,568


,cve_mun,anio_regis
0,01001,2018
1,01001,2018
2,01001,2018
3,01001,2018
4,01003,2018


**4. Contar homicidios por municipio**

In [10]:
# Municipios creados después de 2010 y su municipio padre (clean/hand/mun_crosswalk.yaml)
with open(HAND_DIR / 'mun_crosswalk.yaml', encoding='utf-8') as file:
    new_muns = pd.DataFrame(yaml.safe_load(file)['new_municipalities'])

# Reemplazamos la clave de cada municipio nuevo por la de su padre
parent_of = dict(zip(new_muns['cve_mun_new'], new_muns['cve_mun_2010']))
homicides_clean['cve_mun'] = homicides_clean['cve_mun'].replace(parent_of)

In [11]:
# Homicidios por municipio (solo los municipios con al menos un homicidio; los ceros se agregan en replicate)
homicides_mun = (
    homicides_clean.groupby('cve_mun').size()
    .rename('homicides')
    .reset_index()
)

# Revisión: el total coincide con los homicidios que se conservan
assert homicides_mun['homicides'].sum() == len(homicides_clean)

homicides_mun.head()

,cve_mun,homicides
0,01001,55
1,01002,5
2,01003,4
3,01005,5
4,01006,3


**5. Guardar resultados**

In [12]:
homicides_mun.to_csv(OUTPUT_DIR / '1_clean_homicides.csv', index=False)